# GDBN v9: Do Fundamentals and CEO Data Help? (Point-in-Time)

**Question:** does adding company **fundamentals** (balance sheet, earnings) and **CEO** information improve the best model so far (the slow-input GDBN, notebook 10)?

**Data** (`build_fundamentals.py`, each value used only from the day it was public): book-to-market, gross profitability, leverage, asset growth, earnings surprise (SUE), log CEO pay, equity share of CEO pay, CEO ownership (only from about 2010), CEO tenure, new-CEO flag.

**Why not just add them as GDBN inputs?** The GDBN fits one regression **per stock over time** and drops each stock's average, so it could only learn "when *this* stock gets cheaper, it does better." The known effects of fundamentals compare **across** stocks ("cheap stocks beat expensive stocks"). So fundamentals get their own model:

| Model | What it is |
|---|---|
| **A: baseline** | slow-input GDBN, 1-day target (notebook 10) |
| **F: fundamentals** | one **pooled** regression across all universe stocks each quarter (past 3 years): tomorrow's excess return on each stock's **rank** (0 = lowest, 1 = highest that day) on the 10 features |
| **B: combined** | $z(A) + w \cdot z(F)$, where $z$ = each day's cross-sectional z-score; **$w$ chosen on 2006–2020** (like λ); $w = 0$ means fundamentals do not help |

All portfolios: top/bottom 20%, **held 5 days**, 5 bps costs, same periods (2006–2020 walk-forward; 2021–2022 validation). **Test years are not used.** Takes several minutes (the GDBN walk-forward).

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import pit_common as pc
from gdbn_utils import RESULTS

D = pc.load()
HOLD, H_IC = 5, 5                  # hold 5 days; judge rankings by the 5-day return
FEATURES = ["bm", "gp", "lev", "ag", "sue", "log_pay", "eq_share", "own", "tenure", "new_ceo"]
W_GRID = [0, 0.25, 0.5, 1, 2]

ever = sorted({p for U in D.U.values() for p in U})
fund = pd.read_csv(pc.BUILT / "fundamentals_daily.csv.gz", parse_dates=["date"])
fund = fund[fund["permno"].isin(ever)]
FEAT = {}
for c in FEATURES:
    w = fund.pivot(index="date", columns="permno", values=c).reindex(D.days)
    w.loc[w.index > pd.Timestamp(pc.EVAL_END)] = np.nan            # test years locked
    FEAT[c] = w
del fund
print(f"{len(FEATURES)} features for {len(ever)} stocks ever in the universe")

## 1. A: the baseline (slow-input GDBN)

Same as notebook 10: slow inputs, 1-day target, shared λ chosen by rank IC on 2006–2020.

In [ ]:
res_A = pc.walk_forward(D, pc.slow_features, horizon=1, label="A: slow-input GDBN")
k_A, _ = pc.choose_lambda(D, res_A, 1)
score_A = pc.scores(res_A, k_A)
print(f"λ = {pc.ALPHAS[k_A]:.4f}")

## 2. F: the pooled fundamentals model

Each quarter: on the past 3 years, pool every (day, stock) in the universe and regress tomorrow's excess return on the stock's ranks (that day, among the universe) of the 10 features, as known the day before. Missing values get the middle rank (0.5). Then predict every day of the quarter.

In [ ]:
def ranks(span, U):
    # Each feature as known BEFORE each day, ranked among the universe that day (0 to 1; missing = 0.5)
    return {c: FEAT[c].reindex(index=span, columns=U).shift(1).rank(axis=1, pct=True).fillna(0.5) for c in FEATURES}

parts, coefs = [], []
for j, r in enumerate(D.rebal):
    nxt = D.rebal[j + 1] if j + 1 < len(D.rebal) else pd.Timestamp(pc.EVAL_END) + pd.Timedelta(days=1)
    pos = D.days.searchsorted(r)
    if pos - pc.WINDOW - pc.LOOKBACK < 0:
        continue
    hold = D.days[pos: D.days.searchsorted(nxt)]
    hold = hold[hold <= pd.Timestamp(pc.EVAL_END)]
    if len(hold) == 0:
        continue
    win = D.days[pos - pc.WINDOW: pos]
    U = D.U[r]
    span = win.append(hold)
    R = D.RET.reindex(index=span, columns=U)
    EX = R.sub(R.mean(axis=1), axis=0)
    RK = ranks(span, U)
    Xw = np.column_stack([RK[c].loc[win].values.ravel() for c in FEATURES])
    yw = EX.loc[win].values.ravel()
    ok = ~np.isnan(yw)
    lr = LinearRegression().fit(Xw[ok], yw[ok])
    coefs.append(pd.Series(lr.coef_, index=FEATURES, name=r))
    Xh = np.column_stack([RK[c].loc[hold].values.ravel() for c in FEATURES])
    parts.append(pd.DataFrame((Xh @ lr.coef_).reshape(len(hold), len(U)), index=hold, columns=U))
score_F = pd.concat(parts).sort_index().reindex(index=score_A.index, columns=score_A.columns).where(score_A.notna())
coefs = pd.DataFrame(coefs)
print(f"Fitted {len(coefs)} quarters")

**What the fundamentals model learned:** the average weight on each feature's rank across quarters (in daily excess-return units: +0.0005 = the top-ranked stock earns about 0.05% a day more than the bottom-ranked one), and the share of quarters it was positive. A consistent sign (share near 0 or 1) means a stable effect.

In [ ]:
summary_coef = pd.DataFrame({"avg weight (bps/day, top vs bottom rank)": coefs.mean() * 1e4,
                             "share of quarters > 0": (coefs > 0).mean()}).round(3)
summary_coef

## 3. B: combine, and choose the weight $w$ on 2006–2020

In [ ]:
def zscore(s):
    return s.sub(s.mean(axis=1), axis=0).div(s.std(axis=1).replace(0, np.nan), axis=0)

zA, zF = zscore(score_A), zscore(score_F)
target = pc.forward_excess(D, score_A, H_IC)
ic_by_w = {}
for w in W_GRID:
    ic_by_w[w] = pc.rank_ic((zA + w * zF.fillna(0)).loc[:pc.TUNE_END], target.loc[:pc.TUNE_END]).mean()
W_BEST = max(ic_by_w, key=ic_by_w.get)
print("2006–2020 IC (5-day) by weight on fundamentals:", {k: round(v, 4) for k, v in ic_by_w.items()})
print(f"Chosen w = {W_BEST}" + ("  (fundamentals do not help the GDBN)" if W_BEST == 0 else ""))
score_B = zA + W_BEST * zF.fillna(0)

## 4. Results

Each model held 5 days. **IC t** uses Newey–West standard errors. **ΔIC vs A** is the paired difference in daily IC between the model and the baseline, with its t-statistic: $|t| > 2$ = a significant improvement.

In [ ]:
models = {"A: slow-input GDBN (baseline)": score_A, "F: fundamentals (pooled)": score_F,
          f"B: A + {W_BEST} × F": score_B}
for c in ["sue", "bm", "gp"]:                                    # single known effects as simple benchmarks
    models[f"Benchmark: {c} rank alone"] = FEAT[c].shift(1).reindex(index=score_A.index, columns=score_A.columns).where(score_A.notna())
tables = {}
for period, (a, b) in pc.PERIODS.items():
    ic_A = pc.rank_ic(score_A.loc[a:b], target.loc[a:b])
    rows = {}
    for name, s in models.items():
        ic = pc.rank_ic(s.loc[a:b], target.loc[a:b])
        row = pc.summary(D, pc.held_weights(s, HOLD), a, b, lags=10)
        row["IC (5-day)"], row["IC t"] = ic.mean(), pc.nw_t(ic.values, lags=10)
        d = (ic - ic_A).dropna()
        row["ΔIC vs A"], row["ΔIC t"] = (d.mean(), pc.nw_t(d.values, lags=10)) if not name.startswith("A:") else (np.nan, np.nan)
        rows[name] = row
    tables[period] = pd.DataFrame(rows).T
    print(f"===== {period} =====")
    print(tables[period][["IC (5-day)", "IC t", "ΔIC vs A", "ΔIC t", "Sharpe gross", "Sharpe net", "turnover/day", "breakeven bps"]]
          .round(3).to_string(), "\n")

**Reading it:**
- **F alone:** do the fundamentals rank stocks on their own? (Compare with the single-feature benchmarks: is it just one known effect?)
- **B vs A:** if $w > 0$ was chosen **and** ΔIC is significant **and** it holds in 2021–2022, the fundamentals and CEO data **add information** beyond the GDBN.
- **Turnover:** fundamentals change slowly, so adding them should not raise trading costs much.

## 5. Save results

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
pd.concat(tables, names=["period", "model"]).to_csv(RESULTS / "pit_fundamentals_results.csv")
coefs.to_csv(RESULTS / "pit_fundamentals_coefs.csv")
pd.Series({"w_best": W_BEST, "alpha_A": pc.ALPHAS[k_A], **{f"ic_w_{k}": v for k, v in ic_by_w.items()}}).to_csv(RESULTS / "pit_fundamentals_settings.csv")
print("Saved to", RESULTS)